### **Ensemble (앙상블)**

- 주어진 자료로부터 여러 개의 예측 모형을 만든 후 그것들을 조합하여 하나의 모형을 생성하는 과정
- 대표적인 기법: 배깅, 부스팅



#### **배깅**

- 주어진 자료를 모집단으로 간주하여 주어진 자료에서 여러 개의 붓스트랩 자료를 생성, 각각의 자료에 예측 모형을 만든 후 결합하여 최종 모델을 완성시키는 기법

- 장점
    - 분산을 줄이고 과적합의 문제를 완화
    - 안정적인 성능 보장
    - 비선형 구조의 데이터와 노이즈 데이터에 대한 문제가 완화

- parameter
    - `estimator`
        - 기본값 : None
        - 기본 모델을 설정
    
    - `n_estimator`
        - 기본값 : 10
        - 생성시킬 모델의 개수
    
    - `max_samples`
        - 기본값 : 1.0
        - 생성된 모델들이 사용할 데이터의 개수(인덱스의 수) 비율
    
    - `max_features`
        - 기본값 : 1.0
        - 생성된 모델들이 사용할 feature(column)의 수 비율
    
    - `oob_score`
        - 기본값 : False
        - oob(Out-Of-Bag) 데이터로 일반화 성능을 평가할 것인가?
    
    - `bootstrap`
        - 기본값 : True
        - 샘플링 데이터 이용 시 중복 데이터 허용
        - False인 경우, 각각의 모델들에 `max_samples`의 비율이 1.0이라면 몯 같은 데이터를 학습으로 사용
            - 다양성이 부족해질 수 있음
    
    - `bootstrap_features`
        - 기본값 : False
        - 샘플링 데이터 이용시 중복 column을 허용할 것인가?
    
    - `n_jobs`
        - 기본값 : None
        - CPU의 병렬처리 개수 (-1을 사용하면 모든 CPU를 사용)
    

- 속성
    - `estimators_`
        - 학습된 모델의 리스트

    - `estimators_samples_`
        - 각 모델이 학습한 샘플의 인덱스(행의 위치)

    - `estimators_features_`
        - 각 모델이 학습한 컬럼의 인덱스(열의 위치)

    - `oob_score_`
        - oob 데이터를 기반으로 정확도(분류) / R2(회귀)
    
    - `oob_decision_function_`
        - oob 데이터의 클래스별 예측 확률


- 메서드
    - `fit_predict()`
        - 학습 후 예측 수행(학습 데이터로 예측)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import f1_score

In [ ]:
hotel = pd.read_csv('../data/hotel_bookings.csv')
hotel.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     20000 non-null  int64  
 1   deposit_type                    20000 non-null  str    
 2   lead_time                       19995 non-null  float64
 3   stays_in_weekend_nights         20000 non-null  int64  
 4   stays_in_week_nights            20000 non-null  int64  
 5   is_repeated_guest               19642 non-null  float64
 6   previous_cancellations          20000 non-null  int64  
 7   previous_bookings_not_canceled  20000 non-null  int64  
 8   booking_changes                 20000 non-null  int64  
 9   days_in_waiting_list            20000 non-null  int64  
 10  adr                             18937 non-null  float64
dtypes: float64(3), int64(7), str(1)
memory usage: 1.7 MB


In [4]:
hotel.describe()

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr
count,20000.00000,19995.000000,20000.000000,20000.000000,19642.000000,20000.000000,20000.000000,20000.000000,20000.000000,18937.000000
mean,0.12000,85.978345,0.892550,2.380400,0.038133,0.032900,0.169050,0.269400,1.983950,101.410239
std,0.32497,96.427240,0.952077,1.777345,0.191521,0.455552,1.502426,0.687566,15.927212,49.245097
min,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-6.380000
25%,0.00000,11.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,68.800000
50%,0.00000,51.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,94.500000
75%,0.00000,132.000000,2.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,126.000000
max,1.00000,629.000000,13.000000,30.000000,1.000000,26.000000,66.000000,17.000000,379.000000,451.500000


In [5]:
flag = hotel['adr'] < 0
hotel.loc[flag,]

,is_canceled,deposit_type,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr
1056,0,No Deposit,195.0,4,6,1.0,0,2,2,0,-6.38


In [6]:
hotel = hotel.loc[~flag,]
hotel.describe()

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr
count,19999.000000,19994.000000,19999.000000,19999.000000,19641.000000,19999.000000,19999.000000,19999.000000,19999.000000,18936.000000
mean,0.120006,85.972892,0.892395,2.380219,0.038084,0.032902,0.168958,0.269313,1.984049,101.415931
std,0.324977,96.426569,0.951847,1.777205,0.191403,0.455564,1.502408,0.687474,15.927604,49.240166
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,11.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,68.822500
50%,0.000000,51.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,94.500000
75%,0.000000,132.000000,2.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,126.000000
max,1.000000,629.000000,13.000000,30.000000,1.000000,26.000000,66.000000,17.000000,379.000000,451.500000


In [7]:
# lead_time, adr column의 결측치들은 평균값 또는 중앙값으로 대체

hotel['lead_time'] = hotel['lead_time'].fillna( hotel['lead_time'].mean() )
hotel['adr'] = hotel['adr'].fillna( hotel['adr'].mean() )

In [10]:
hotel['is_repeated_guest'].mode()[0]

np.float64(0.0)

In [11]:
hotel['is_repeated_guest'] = hotel['is_repeated_guest'].fillna(
    hotel['is_repeated_guest'].mode()[0]
)

In [12]:
# deposit_type column은 문자형 데이터 → 문자의 유형들을 확인
hotel['deposit_type'].unique()

<StringArray>
['No Deposit', 'Refundable', 'Non Refund']
Length: 3, dtype: str

In [13]:
hotel['deposit_type'].value_counts()

deposit_type
No Deposit    19137
Non Refund      834
Refundable       28
Name: count, dtype: int64

In [14]:
df = pd.get_dummies(hotel, columns=['deposit_type'], drop_first = True)

In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 19999 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     19999 non-null  int64  
 1   lead_time                       19999 non-null  float64
 2   stays_in_weekend_nights         19999 non-null  int64  
 3   stays_in_week_nights            19999 non-null  int64  
 4   is_repeated_guest               19999 non-null  float64
 5   previous_cancellations          19999 non-null  int64  
 6   previous_bookings_not_canceled  19999 non-null  int64  
 7   booking_changes                 19999 non-null  int64  
 8   days_in_waiting_list            19999 non-null  int64  
 9   adr                             19999 non-null  float64
 10  deposit_type_Non Refund         19999 non-null  bool   
 11  deposit_type_Refundable         19999 non-null  bool   
dtypes: bool(2), float64(3), int64(7)
memory usage: 1

In [16]:
df['is_canceled'].value_counts()

is_canceled
0    17599
1     2400
Name: count, dtype: int64

In [ ]:
X = df.drop('is_canceled', axis = 1)
y = df['is_canceled']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

In [21]:
y_train.value_counts()

is_canceled
0    14079
1     1920
Name: count, dtype: int64

In [46]:
# 배깅 모델 생성 → DecisionTree를 기본 모델로 사용 → 데이터의 불균형이 심하다 → class_weight
base_model = DecisionTreeClassifier(class_weight = 'balanced', max_depth=3)
model = BaggingClassifier(base_model, n_estimators = 100)

In [47]:
model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.","DecisionTreeC..., max_depth=3)"
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [48]:
pred = model.predict(X_test)

In [49]:
print( round( f1_score(y_test, pred), 4 ) )

0.5748


- 데이터 불균형 해소

1. RandomOverSampling / 2:1로 resampling
2. Bagging model의 bootstrap을 0.8로 지정하고 n_estimators를 100으로 설정
    - DecisionTree Classifier의 class_weight는 default 값으로, max_depth는 3으로 설정

In [50]:
from imblearn.over_sampling import RandomOverSampler

In [55]:
ros = RandomOverSampler(sampling_strategy=0.5)
over_X, over_y = ros.fit_resample(X, y)
over_y.value_counts()

is_canceled
0    17599
1     8799
Name: count, dtype: int64

In [77]:
base_model = DecisionTreeClassifier(max_depth = 3)
model = BaggingClassifier(max_samples = 0.8, n_estimators = 100)

In [69]:
X_train, X_test, y_train, y_test = train_test_split(
    over_X, over_y, test_size = 0.2, random_state = 42, stratify = over_y
)

In [70]:
model.fit(X_train, y_train)
pred = model.predict(X_test)
print( round( f1_score(y_test, pred), 4 ) )

0.9326


In [72]:
# SMOTE 방식으로 오버 샘플링, 데이터는 1:1

from imblearn.over_sampling import SMOTE

smote = SMOTE()
X_sm, y_sm = smote.fit_resample(X, y)

In [73]:
y_sm.value_counts()

is_canceled
0    17599
1    17599
Name: count, dtype: int64

In [90]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sm, y_sm, test_size = 0.2, random_state = 42, stratify = y_sm
)

In [91]:
base_model = DecisionTreeClassifier()
clf2 = BaggingClassifier(estimator=base_model, max_samples = 0.8, n_estimators = 100)
clf2.fit(X_train, y_train)

clf3 = BaggingClassifier(max_samples = 0.8, n_estimators = 100)
clf3.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.8
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [92]:
pred = clf2.predict(X_test)
pred2 = clf3.predict(X_test)
print( round( f1_score(y_test, pred), 4 ), round( f1_score(y_test, pred2), 4 ) )

0.9138 0.9134


In [93]:
# 기본 decisiontree 학습

base_model.fit(X_train, y_train)
pred3 = base_model.predict(X_test)
print(round( f1_score(y_test, pred3), 4 ))

0.8852


In [94]:
clf_oob = BaggingClassifier(estimator=base_model, n_estimators=100, oob_score=True)
clf_oob.fit(over_X, over_y)
clf_oob.oob_score_

0.9615879990908403

In [95]:
# bagging 회귀 → bagging용 회귀 모델 로드 → 기본 모델은 decisiontree(회귀)

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor

In [96]:
car = pd.read_csv('../data/CarPrice_Assignment.csv')
car.info()

<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   CarName           205 non-null    str    
 3   fueltype          205 non-null    str    
 4   aspiration        205 non-null    str    
 5   doornumber        205 non-null    str    
 6   carbody           205 non-null    str    
 7   drivewheel        205 non-null    str    
 8   enginelocation    205 non-null    str    
 9   wheelbase         205 non-null    float64
 10  carlength         205 non-null    float64
 11  carwidth          205 non-null    float64
 12  carheight         205 non-null    float64
 13  curbweight        205 non-null    int64  
 14  enginetype        205 non-null    str    
 15  cylindernumber    205 non-null    str    
 16  enginesize        205 non-null    int64  
 17  fuelsyst

In [103]:
df_str = car.select_dtypes('object')
df_int = car.select_dtypes('number')

C:\Users\hkssn\AppData\Local\Temp\ipykernel_7372\3080780290.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_str = car.select_dtypes('object')


In [104]:
df_int.describe()

,car_ID,symboling,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
count,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000
mean,103.000000,0.834146,98.756585,174.049268,65.907805,53.724878,2555.565854,126.907317,3.329756,3.255415,10.142537,104.117073,5125.121951,25.219512,30.751220,13276.710571
std,59.322565,1.245307,6.021776,12.337289,2.145204,2.443522,520.680204,41.642693,0.270844,0.313597,3.972040,39.544167,476.985643,6.542142,6.886443,7988.852332
min,1.000000,-2.000000,86.600000,141.100000,60.300000,47.800000,1488.000000,61.000000,2.540000,2.070000,7.000000,48.000000,4150.000000,13.000000,16.000000,5118.000000
25%,52.000000,0.000000,94.500000,166.300000,64.100000,52.000000,2145.000000,97.000000,3.150000,3.110000,8.600000,70.000000,4800.000000,19.000000,25.000000,7788.000000
50%,103.000000,1.000000,97.000000,173.200000,65.500000,54.100000,2414.000000,120.000000,3.310000,3.290000,9.000000,95.000000,5200.000000,24.000000,30.000000,10295.000000
75%,154.000000,2.000000,102.400000,183.100000,66.900000,55.500000,2935.000000,141.000000,3.580000,3.410000,9.400000,116.000000,5500.000000,30.000000,34.000000,16503.000000
max,205.000000,3.000000,120.900000,208.100000,72.300000,59.800000,4066.000000,326.000000,3.940000,4.170000,23.000000,288.000000,6600.000000,49.000000,54.000000,45400.000000


In [107]:
df_str.head()

,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,enginetype,cylindernumber,fuelsystem
0,alfa-romero giulia,gas,std,two,convertible,rwd,front,dohc,four,mpfi
1,alfa-romero stelvio,gas,std,two,convertible,rwd,front,dohc,four,mpfi
2,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,ohcv,six,mpfi
3,audi 100 ls,gas,std,four,sedan,fwd,front,ohc,four,mpfi
4,audi 100ls,gas,std,four,sedan,4wd,front,ohc,five,mpfi


In [106]:
df_str['doornumber'].unique()

<StringArray>
['two', 'four']
Length: 2, dtype: str

In [108]:
df_str['doornumber'] = df_str['doornumber'].map(
    lambda x:2 if x=='two' else 4
)

In [109]:
df_str['cylindernumber'].unique()

<StringArray>
['four', 'six', 'five', 'three', 'twelve', 'two', 'eight']
Length: 7, dtype: str

In [110]:
df_str['cylindernumber'] = df_str['cylindernumber'].map(
    {
        'four': 4,
        'six': 6,
        'five': 5,
        'three': 3,
        'twelve': 12,
        'two': 2,
        'eight': 8
    }
)

In [111]:
df_str.info()

<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   CarName         205 non-null    str  
 1   fueltype        205 non-null    str  
 2   aspiration      205 non-null    str  
 3   doornumber      205 non-null    int64
 4   carbody         205 non-null    str  
 5   drivewheel      205 non-null    str  
 6   enginelocation  205 non-null    str  
 7   enginetype      205 non-null    str  
 8   cylindernumber  205 non-null    int64
 9   fuelsystem      205 non-null    str  
dtypes: int64(2), str(8)
memory usage: 16.1 KB


In [121]:
# 특정 문자의 위치를 찾는다: find() / index()
vals = list(
    map(
        lambda x: [ x[ : x.find(' ')] , x[x.find(' ') : ].strip()] ,
        df_str['CarName']
    )
)

In [124]:
df_str[ [ 'brand', 'modelName' ] ] = vals
df_str.head()

,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,enginetype,cylindernumber,fuelsystem,brand,modelName
0,alfa-romero giulia,gas,std,2,convertible,rwd,front,dohc,4,mpfi,alfa-romero,giulia
1,alfa-romero stelvio,gas,std,2,convertible,rwd,front,dohc,4,mpfi,alfa-romero,stelvio
2,alfa-romero Quadrifoglio,gas,std,2,hatchback,rwd,front,ohcv,6,mpfi,alfa-romero,Quadrifoglio
3,audi 100 ls,gas,std,4,sedan,fwd,front,ohc,4,mpfi,audi,100 ls
4,audi 100ls,gas,std,4,sedan,4wd,front,ohc,5,mpfi,audi,100ls


In [125]:
# CarName과 modelName은 제거

df_str.drop(['CarName', 'modelName'], axis = 1, inplace=True)

In [126]:
cols = df_str.select_dtypes('object').columns
df_str2 = pd.get_dummies(df_str, columns=cols, drop_first=True)

C:\Users\hkssn\AppData\Local\Temp\ipykernel_7372\1006457860.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols = df_str.select_dtypes('object').columns


In [128]:
df_str2

,doornumber,cylindernumber,fueltype_gas,aspiration_turbo,carbody_hardtop,carbody_hatchback,carbody_sedan,carbody_wagon,drivewheel_fwd,drivewheel_rwd,...,brand_renault,brand_saab,brand_subar,brand_subaru,brand_toyota,brand_toyouta,brand_vokswagen,brand_volkswagen,brand_volvo,brand_vw
0,2,4,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,2,4,True,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,2,6,True,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,4,4,True,False,False,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,4,5,True,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,4,4,True,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
201,4,4,True,True,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
202,4,6,True,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
203,4,6,False,True,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False


In [129]:
df_int.drop('car_ID', axis=1, inplace=True)

In [130]:
df = pd.concat( [df_str2, df_int], axis=1 )

In [131]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 67 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   doornumber           205 non-null    int64  
 1   cylindernumber       205 non-null    int64  
 2   fueltype_gas         205 non-null    bool   
 3   aspiration_turbo     205 non-null    bool   
 4   carbody_hardtop      205 non-null    bool   
 5   carbody_hatchback    205 non-null    bool   
 6   carbody_sedan        205 non-null    bool   
 7   carbody_wagon        205 non-null    bool   
 8   drivewheel_fwd       205 non-null    bool   
 9   drivewheel_rwd       205 non-null    bool   
 10  enginelocation_rear  205 non-null    bool   
 11  enginetype_dohcv     205 non-null    bool   
 12  enginetype_l         205 non-null    bool   
 13  enginetype_ohc       205 non-null    bool   
 14  enginetype_ohcf      205 non-null    bool   
 15  enginetype_ohcv      205 non-null    bool   
 16  e

In [132]:
X = df.drop('price', axis = 1)
y = df['price']

In [133]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42
)

In [134]:
base_model = DecisionTreeRegressor()
reg = BaggingRegressor(estimator=base_model, n_estimators=100, oob_score=True)

In [135]:
reg.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeRegressor`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeRegressor()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",True
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [137]:
reg.oob_score_

0.9101883394986057

In [142]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [140]:
pred = reg.predict(X_test)

In [143]:
print(mean_absolute_error(y_test, pred))
print(mean_squared_error(y_test, pred))
print(r2_score(y_test, pred))

1339.1910161290323
3852710.4811563105
0.9443927973609738
